# 10_filtering_hybrid: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/10_filtering_hybrid/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Fuse sparse lexical matches with a dense ranking.

Concepts: Fuse sparse lexical matches with a dense ranking.
Run: uv run python modules/10_filtering_hybrid/examples/01_from_scratch.py
Expected: Inspect the printed IDs or measurements and the checked invariant.
CPU: seconds on the included small workload. Uses local data; no Docker or model download.
"""

from tinyvec.hybrid import BM25, fuse, maxsim


def main():
    documents = ["waterproof hiking boots SKU-7", "boots for wet trails", "linen shirt"]
    lexical = BM25(documents).rank("SKU-7")
    dense = [1, 0, 2]
    fused = fuse([dense, lexical])
    print("lexical", lexical, "dense", dense, "fused", fused)
    assert fused[0][0] == 0
    # Each query token chooses its own strongest document token.
    assert maxsim([[1, 0], [0, 1]], [[1, 0], [0, 1]]) == 2


if __name__ == "__main__":
    main()

## 02_production

Source: `modules/10_filtering_hybrid/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("faiss") is None:
    print("SKIP native comparison: --extra libs")
else:
    """Retrieve sparse product matches using bm25s.

    Concepts: Retrieve sparse product matches using bm25s.
    Run: uv run --extra libs python modules/10_filtering_hybrid/examples/02_production.py
    Expected: Inspect the printed IDs or measurements and the checked invariant.
    CPU: seconds on the included small workload. Uses local data; no Docker or model download.
    """

    # course-extra: libs
    import bm25s

    from common.data import product_texts
    from common.teaching import catalog

    def main():
        products, _, _, _ = catalog(200)
        corpus = product_texts(products)
        engine = bm25s.BM25()
        engine.index(bm25s.tokenize(corpus, show_progress=False), show_progress=False)
        rows, scores = engine.retrieve(
            bm25s.tokenize(["waterproof hiking boots"], show_progress=False),
            k=5,
            show_progress=False,
        )
        print(list(zip(products.iloc[rows[0]].title, scores[0].tolist(), strict=True)))
        assert scores[0][0] > 0

    if __name__ == "__main__":
        main()

## 03_measure

Source: `modules/10_filtering_hybrid/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Measure strict filtering and score a second-stage ranking.

Concepts: Measure strict filtering and score a second-stage ranking.
Run: uv run python modules/10_filtering_hybrid/examples/03_measure.py
Expected: Inspect the printed IDs or measurements and the checked invariant.
CPU: seconds on the included small workload. Uses local data; no Docker or model download.
"""

import json

from common.bench.metrics import ndcg_at_k
from common.data import search_queries
from common.index_lessons import trial
from common.settings import ROOT
from common.teaching import evidence


def main():
    ranking = json.loads((ROOT / "data/tier2/reranking.json").read_text())
    judged = next(q for q in search_queries() if q["id"] == ranking["query_id"])
    after = sorted(ranking["candidates"], key=lambda r: -r["score"])
    observations = {
        "filter": [
            trial(
                "hnsw",
                expression={"op": "eq", "field": "region", "value": "north"},
                filter_policy=p,
                oversample=1,
            )
            for p in ("post", "traverse", "adaptive")
        ],
        "cross_encoder": after,
        "before": [r["id"] for r in ranking["candidates"]],
        "ndcg_before": ndcg_at_k([r["id"] for r in ranking["candidates"]], judged["qrels"], 10),
        "ndcg_after": ndcg_at_k([r["id"] for r in after], judged["qrels"], 10),
    }
    print(observations)
    evidence("lesson-10_filtering_hybrid", {"observations": observations}, tier=2)


if __name__ == "__main__":
    main()

## 04_lance_hybrid

Source: `modules/10_filtering_hybrid/examples/04_lance_hybrid.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("chromadb") is None:
    print("SKIP native comparison: --extra embedded")
else:
    """Store token vectors and rerank native LanceDB candidates.

    Run: uv run --extra embedded python modules/10_filtering_hybrid/examples/04_lance_hybrid.py
    Expected: compare the reported ranking and verify the final assertion.
    CPU: seconds, included data, no Docker or model downloads.
    """

    # course-extra: embedded
    import tempfile

    import lancedb
    import pyarrow as pa
    from lancedb.index import FTS

    from common.index_lessons import unit_catalog
    from tinyvec.hybrid import maxsim, tokens

    def main():
        products, matrix, q = unit_catalog(160)
        # Interpretable token features for this scorer demonstration; neural token encoders use learned features.
        vocabulary = ["mug", "warmer", "coffee", "desk"]

        def features(text):
            return [[float(word == term) for term in vocabulary] for word in tokens(text)] or [
                [0.0] * 4
            ]

        rows = [
            {"id": p.id, "text": p.title, "vector": v.tolist(), "token_vectors": features(p.title)}
            for p, v in zip(products.itertuples(), matrix, strict=True)
        ]
        schema = pa.schema(
            [
                ("id", pa.string()),
                ("text", pa.string()),
                ("vector", pa.list_(pa.float32(), matrix.shape[1])),
                ("token_vectors", pa.list_(pa.list_(pa.float32(), 4))),
            ]
        )
        with tempfile.TemporaryDirectory() as directory:
            db = lancedb.connect(directory)
            table = db.create_table("products", data=pa.Table.from_pylist(rows, schema=schema))
            table.create_index("text", config=FTS())
            hits = (
                table.search(query_type="hybrid", vector_column_name="vector")
                .vector(q)
                .text("mug warmer")
                .limit(8)
                .to_list()
            )
            reranked = sorted(
                hits, key=lambda row: -maxsim(features("mug warmer"), row["token_vectors"])
            )
            print(
                "native hybrid",
                [h["id"] for h in hits],
                "late interaction",
                [h["id"] for h in reranked],
            )
            assert hits and {r["id"] for r in hits} == {r["id"] for r in reranked}
            del table, db

    if __name__ == "__main__":
        main()

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from tinyvec.hybrid import fuse

print(fuse([["sku-9", "boot"], ["boot", "lamp"]], constant=20))

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.